In [1]:
import numpy as np
from pathlib import Path

import rtde_control 
import rtde_receive
import rtde_io
import time 

In [3]:
ROBOT_IP = "192.168.1.102"
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)

# --- READ joint positions (radians, 6 joints) ---
joint_q = rtde_r.getActualQ()
print("Joint positions [rad]:", joint_q)

# --- READ Tool Center Point (TCP) pose ---
# Returns [x, y, z, rx, ry, rz] in meters and axis-angle rotation
tcp_pose = rtde_r.getActualTCPPose()
print("TCP pose [m, rad]:", tcp_pose)

#### save q and tcp_pose to a file 
np.save(Path("robot_state.npy"), {"joint_q": joint_q, "tcp_pose": tcp_pose})

Joint positions [rad]: [-0.395301644002096, -1.3939289164594193, 1.6332433859454554, -2.3888732395567835, -1.3905060927020472, -0.03921825090517217]
TCP pose [m, rad]: [-0.6055859573406256, 0.08748400322866547, 0.4604151957277502, -2.154361012942046, -1.5914912650920099, 0.6296848387954087]


In [ ]:
REGISTER_ADDRESS = 18

print(f"Connecting to robot at {ROBOT_IP}...")
# Initialize the RTDE IO Interface
io_interface = rtde_io.RTDEIOInterface(ROBOT_IP)
print("Connected!")

def open_gripper():
    print("Sending OPEN command...")
    # Sends a '1' to register 18, triggering your "If action_cmd == 1" node
    io_interface.setInputIntRegister(REGISTER_ADDRESS, 1)
    
def close_gripper():
    print("Sending CLOSE command...")
    # Sends a '2' to register 18, triggering your "ElseIf action_cmd == 2" node
    io_interface.setInputIntRegister(REGISTER_ADDRESS, 2)

def reset_gripper_command():
    # Sets it back to 0 so the robot doesn't continuously spam the grip command 
    # on every single loop iteration.
    io_interface.setInputIntRegister(REGISTER_ADDRESS, 0)

# --- Execution Test ---
try:
    # 1. Open the gripper
    open_gripper()
    time.sleep(2) # Wait for the physical movement to finish
    reset_gripper_command() 
    time.sleep(1)

    # 2. Close the gripper
    close_gripper()
    time.sleep(2)
    reset_gripper_command()
    
    print("Test complete.")

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    # Always a good idea to reset before disconnecting
    reset_gripper_command()

io_interface.disconnect()


Connecting to robot at 192.168.1.102...


RuntimeError: One of the RTDE input registers are already in use! Currently you must disable the EtherNet/IP adapter, PROFINET or any MODBUS unit configured on the robot. This might change in the future.

In [ ]:
from onRobot.gripper import RG2

gripper = RG2(0)
print(gripper.get_rg_width())

# 3. Control the Gripper
print("Closing gripper...")
gripper.rg_grip(target_width= 10, target_force= 40)
time.sleep(1.5) # Allow time for the physical action

# 5. Close the gripper
print("Opening gripper...")
gripper.rg_grip(target_width= 100, target_force=20)


78.5
Closing gripper...


ConnectTimeout: HTTPConnectionPool(host='192.168.0.99', port=41414): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='192.168.0.99', port=41414) at 0x106751810>, 'Connection to 192.168.0.99 timed out. (connect timeout=None)'))

RTDEReceiveInterface boost system Exception: (asio.misc:2) End of file [asio.misc:2 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_recv_op.hpp:133:37 in function 'do_complete']
